# Classical 1D Transect Interpolation
This notebook demonstrates classical 1D curve interpolations (Lagrange/Newton Polynomial, Piecewise Linear, and Cubic Spline) applied to 1D Longitude transects of Heavy Metals data from the healed dataset. It produces a visualization to directly expose Runge's phenomenon alongside smooth $C^2$ continuity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, CubicSpline, BarycentricInterpolator

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Define target columns
targets = ['Heavy_Metals_Hg_ug_L', 'Heavy_Metals_Pb_ug_L', 'Heavy_Metals_Cd_ug_L']

## 1. Data Extract & 1D Spatial Slicing
We load the healed dataset (`../output/mse_healed_data.csv`). We will extract a narrow, horizontal "Latitude slice" to construct a true 1D sequence of points where `x = Longitude` and `y = Concentration`.

In [ ]:
df = pd.read_csv('../output/mse_healed_data.csv')

# We select the median latitude as our horizontal 1D transect band
target_lat = df['Latitude'].median()
profile_df = df[(df['Latitude'] >= target_lat - 1.0) & (df['Latitude'] <= target_lat + 1.0)]

# Sort by Longitude so curves evaluate chronologically left-to-right, and remove duplicates perfectly
profile_df = profile_df.sort_values('Longitude').drop_duplicates('Longitude')


## 2. 1D Interpolation Builder
Here we implement:
- **Exact Polynomial (Lagrange/Newton):** Implemented via `BarycentricInterpolator` to calculate the *exact* high-degree theoretical polynomial passing through all nodes, explicitly triggering Runge's Oscillations.
- **Piecewise Linear:** $C^0$ jagged, connected lines passing through all nodes.
- **Cubic Spline:** $C^2$ smooth curved lines passing through all nodes.

In [ ]:
def plot_1d_transects(target_col):
    # Extract X (Longitude) and Y (Target Value) 
    x_all = profile_df['Longitude'].values
    y_all = profile_df[target_col].values
    
    # Subsample nodes so that we have ~15 data points. 
    # High-degree polynomials (>20 points) crash mathematically into infinity. 
    # 15 nodes is perfect to demonstrate interpolation logic and Runge's phenomenon.
    indices = np.linspace(0, len(x_all) - 1, 15, dtype=int)
    x_nodes = x_all[indices]
    y_nodes = y_all[indices]
    
    # Dense X evaluation grid for plotting continuous curves
    x_fine = np.linspace(x_nodes.min(), x_nodes.max(), 500)
    
    # 1. Exact Polynomial Interpolation (Lagrange/Newton formulation via Barycentric matching)
    f_poly = BarycentricInterpolator(x_nodes, y_nodes)
    y_poly = f_poly(x_fine)
    
    # 2. Piecewise Linear Interpolation (C0)
    f_linear = interp1d(x_nodes, y_nodes, kind='linear')
    y_linear = f_linear(x_fine)
    
    # 3. Cubic Spline Interpolation (C2)
    f_cubic = CubicSpline(x_nodes, y_nodes)
    y_cubic = f_cubic(x_fine)
    
    # Visualize the exact layout
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # To prevent visual blow-out from Runge's phenomenon, we bound the plot based on data limits
    y_margin = (y_nodes.max() - y_nodes.min()) * 1.5
    y_upper = y_nodes.max() + y_margin
    y_lower = y_nodes.min() - y_margin
    ax1.set_ylim([max(0, y_lower), y_upper]) # Set lower bound to 0 if values are positive limits

    # Curves
    ax1.plot(x_fine, y_poly, 'r--', label='Polynomial (Runge Oscillations)')
    ax1.plot(x_fine, y_linear, 'b:', label='Linear (C0 - Jagged)')
    ax1.plot(x_fine, y_cubic, 'g-', linewidth=2, label='Cubic Spline (C2 - Smooth)')
    
    # Known Nodes
    ax1.scatter(x_nodes, y_nodes, color='black', zorder=5, label='Data Nodes')
    
    ax1.set_title(f"1D Interpolation Methods Comparison: {target_col}")
    ax1.set_xlabel("Longitude")
    ax1.set_ylabel(f"Concentration ({target_col})")
    ax1.legend()
    
    plt.tight_layout()
    plt.show()

## 3. Generate Visualizations

In [ ]:
for col in targets:
    plot_1d_transects(col)